# 01.02 · NumPy & vectors

**Chapter:** Python & Math for AI · **Milestone:** M1

## Goals
- [ ] Create, index and reshape arrays; read `.shape` fluently
- [ ] Understand broadcasting and why vectorised code is fast
- [ ] Compute dot product, norm and cosine similarity — the core of semantic search

## Resources
- [NumPy for absolute beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) — free
- [3Blue1Brown – Essence of linear algebra, ch. 1-2 & 9](https://www.3blue1brown.com/topics/linear-algebra) — free

Every model input, weight and output is an array of numbers. An **embedding** (chapter 07) is just a vector;
"find similar documents" is "find vectors pointing in a similar direction".

In [2]:
import numpy as np

rng = np.random.default_rng(0)

## 1. Arrays and shapes
`shape` is the single most important attribute. Most deep-learning bugs are shape bugs.

In [3]:
v = np.array([1.0, 2.0, 3.0])  # vector, shape (3,)
M = np.arange(12).reshape(3, 4)  # matrix, shape (3, 4)
batch = rng.normal(size=(8, 16, 64))  # e.g. (batch, sequence, features) — the shape of transformer activations

print(v.shape, M.shape, batch.shape, batch.dtype)
print(M)
print("row 1:", M[1], "| column 2:", M[:, 2], "| last 2 cols:\n", M[:, -2:])
print("first sequence, first token, first 5 features:", batch[0, 0, :5].round(2))

(3,) (3, 4) (8, 16, 64) float64
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
row 1: [4 5 6 7] | column 2: [ 2  6 10] | last 2 cols:
 [[ 2  3]
 [ 6  7]
 [10 11]]
first sequence, first token, first 5 features: [ 0.13 -0.13  0.64  0.1  -0.54]


## 2. Vectorisation and broadcasting
Operations apply element-wise across whole arrays, in C, without Python loops.
**Broadcasting**: when shapes differ, NumPy stretches size-1 (or missing) dimensions to match.
Rule: compare shapes from the right; dimensions must be equal or 1.

In [4]:
x = rng.normal(size=(1000, 384))  # 1000 "embeddings" of dimension 384

mean = x.mean(axis=0)  # shape (384,)  — mean of each feature
std = x.std(axis=0)  # shape (384,)
x_norm = (x - mean) / std  # (1000, 384) - (384,) broadcasts across rows
print(mean.shape, x_norm.shape, x_norm.mean(axis=0)[:3].round(6), x_norm.std(axis=0)[:3].round(3))

(384,) (1000, 384) [ 0. -0.  0.] [1. 1. 1.]


In [5]:
import time

def py_sum_squares(a):
    total = 0.0
    for row in a:
        for val in row:
            total += val * val
    return total

start = time.perf_counter(); py_sum_squares(x); t_py = time.perf_counter() - start
start = time.perf_counter(); (x**2).sum(); t_np = time.perf_counter() - start
print(f"python loops: {t_py*1000:.1f} ms | numpy: {t_np*1000:.3f} ms | speed-up ≈ {t_py/t_np:.0f}x")

python loops: 31.1 ms | numpy: 0.617 ms | speed-up ≈ 50x


## 3. Dot product, norm, cosine similarity
- **Dot product** $a \cdot b = \sum_i a_i b_i$: large when vectors point the same way *and* are long.
- **Norm** $\|a\| = \sqrt{a \cdot a}$: the length.
- **Cosine similarity** $\cos\theta = \frac{a \cdot b}{\|a\|\|b\|}$: only direction matters. Range $[-1, 1]$.

Embedding search almost always uses cosine (or dot product on pre-normalised vectors, which is the same thing).

In [6]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])  # same direction, twice as long
c = np.array([-3.0, 0.0, 1.0])  # perpendicular to a

def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

print("a·b =", a @ b, "| cos(a,b) =", round(cosine(a, b), 3), "| cos(a,c) =", round(cosine(a, c), 3))

a·b = 28.0 | cos(a,b) = 1.0 | cos(a,c) = 0.0


## 4. A toy semantic search
Hand-made 4-D "embeddings" with dimensions roughly meaning *[animal, vehicle, food, tech]*.
Real embeddings have 384-3072 dimensions learned by a model, but the search is identical.

In [7]:
docs = ["dog", "cat", "car", "truck", "pizza", "laptop", "puppy", "burger"]
E = np.array([
    [0.9, 0.0, 0.1, 0.0],  # dog
    [0.8, 0.0, 0.1, 0.1],  # cat
    [0.0, 0.9, 0.0, 0.2],  # car
    [0.0, 1.0, 0.0, 0.1],  # truck
    [0.0, 0.0, 1.0, 0.0],  # pizza
    [0.0, 0.1, 0.0, 0.9],  # laptop
    [0.95, 0.0, 0.05, 0.0],  # puppy
    [0.1, 0.0, 0.9, 0.0],  # burger
])

def search(query_vec, k=3):
    E_unit = E / np.linalg.norm(E, axis=1, keepdims=True)  # normalise every row once
    q_unit = query_vec / np.linalg.norm(query_vec)
    scores = E_unit @ q_unit  # (8, 4) @ (4,) -> (8,): cosine to every doc in one matmul
    top = np.argsort(-scores)[:k]
    return [(docs[i], round(float(scores[i]), 3)) for i in top]

print(search(np.array([1.0, 0.0, 0.0, 0.0])))  # "animal"
print(search(np.array([0.0, 0.5, 0.0, 0.5])))  # "vehicle + tech" -> car/laptop

[('puppy', 0.999), ('dog', 0.994), ('cat', 0.985)]
[('car', 0.844), ('laptop', 0.781), ('truck', 0.774)]


## Exercises
1. Write `cosine_matrix(A, B)` returning all pairwise cosine similarities (shape `(len(A), len(B))`) **without loops**.
   Check against `sklearn.metrics.pairwise.cosine_similarity`.
2. Write `top_k(A, B, k)` returning, for each row of A, the indices of the k most similar rows of B.
3. Time your function on A=(1000, 384), B=(50000, 384). How would this scale to 10M documents? (Chapter 07 answers: ANN indexes.)

In [8]:
# Exercise 1:
from sklearn.metrics.pairwise import cosine_similarity

def cosine_matrix(A, B):
    dot = A @ B.T
    norm = np.linalg.norm(A, axis=1, keepdims=True) * np.linalg.norm(B, axis=1, keepdims=True).T
    return dot / norm

# Exercise 2:
def top_k(A, B, k=3):
    scores = cosine_matrix(A, B)
    top = np.argsort(-scores, axis=1)[:, :k]
    return top

# Exercise 1 results:
A, B = rng.normal(size=(5, 8)), rng.normal(size=(7, 8))
assert np.allclose(cosine_matrix(A, B), cosine_similarity(A, B))

# Exercise 2 results:
A, B = rng.normal(size=(5, 8)), rng.normal(size=(7, 8))
print(top_k(A, B))

# Exercise 3:
A, B = rng.normal(size=(1000, 384)), rng.normal(size=(50000, 384))
start = time.perf_counter()
top_k(A, B)
print(f"time: {time.perf_counter() - start:.2f}s")

# At 10M documents, this takes ~570s taking 80GB RAM. Impossible to do in-memory, chapter 07 will cover ANN indexes.
# A, B = rng.normal(size=(1000000, 384)), rng.normal(size=(50000, 384))
# start = time.perf_counter()
# top_k(A, B)
# print(f"time: {time.perf_counter() - start:.2f}s")


[[2 1 6]
 [0 1 6]
 [3 4 5]
 [1 2 0]
 [6 0 1]]
time: 3.02s
